# Tushare 数据探索

验证 Tushare API 连通性并探索公告相关数据接口。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import tushare as ts
import pandas as pd
import os
from dotenv import load_dotenv
from src.config import config

load_dotenv()

# 设置显示
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

In [ ]:
# 连接 Tushare
token = os.getenv("TUSHARE_TOKEN")
print(f"Token: {'***' + token[-4:] if token else 'NOT SET'}")

ts.set_token(token)
pro = ts.pro_api()
print("Tushare Pro API ready")

In [ ]:
# 1. 获取全部 A 股列表
df_stocks = pro.stock_basic(
    exchange='',
    list_status='L',
    fields='ts_code,symbol,name,area,industry,list_date'
)
print(f"Total listed stocks: {len(df_stocks)}")
print(f"Industries: {df_stocks['industry'].nunique()}")
df_stocks.head(10)

In [ ]:
# 行业分布
industry_counts = df_stocks['industry'].value_counts().head(20)
industry_counts

In [ ]:
# 2. 试点行业股票筛选
pilot_keywords = ['电气设备', '新能源', '光伏', '风电', '电池', '电力设备']
pilot_mask = df_stocks['industry'].str.contains('|'.join(pilot_keywords), na=False)
df_pilot = df_stocks[pilot_mask]
print(f"Pilot industry stocks: {len(df_pilot)}")
df_pilot

In [ ]:
# 3. 定期报告披露时间表
# 拉取 2024 年全年的定期报告披露安排
df_disclosure = pro.disclosure(
    start_date='20240101',
    end_date='20241231'
)
print(f"Total disclosure records: {len(df_disclosure)}")
if not df_disclosure.empty:
    print(f"Columns: {list(df_disclosure.columns)}")
    df_disclosure.head(10)

In [ ]:
# 披露时间表统计
if not df_disclosure.empty:
    # 按 end_date（报告期）统计
    if 'end_date' in df_disclosure.columns:
        df_disclosure['report_period'] = df_disclosure['end_date'].str[:4]
        print("Reports by period:")
        print(df_disclosure['report_period'].value_counts().sort_index())
    
    # 有多少条有实际披露日期？
    if 'actual_date' in df_disclosure.columns:
        has_actual = df_disclosure['actual_date'].notna().sum()
        print(f"\nWith actual disclosure date: {has_actual}/{len(df_disclosure)}")

In [ ]:
# 4. 财务数据测试（拉取宁德时代的利润表）
df_income = pro.income(
    ts_code='300750.SZ',
    report_type='1',  # 1=合并报表
    start_date='20230101',
    end_date='20241231'
)
print(f"Income statement records: {len(df_income)}")
if not df_income.empty:
    # 显示最重要的列
    key_cols = ['ts_code', 'end_date', 'revenue', 'n_income', 'total_revenue', 
                'oper_cost', 'sell_exp', 'admin_exp']
    available = [c for c in key_cols if c in df_income.columns]
    df_income[available].head()

In [ ]:
# 5. 日线行情 — 验证公告前后价格变动
df_daily = pro.daily(
    ts_code='300750.SZ',
    start_date='20240401',
    end_date='20240430'
)
print(f"Daily records: {len(df_daily)}")
if not df_daily.empty:
    print(f"Columns: {list(df_daily.columns)}")
    cols = ['trade_date', 'open', 'high', 'low', 'close', 'vol', 'amount', 'pct_chg']
    available = [c for c in cols if c in df_daily.columns]
    df_daily[available].head(5)

In [ ]:
# 6. 东方财富公告接口测试
import requests

url = "https://np-anotice-stock.eastmoney.com/api/security/ann"
params = {
    "page_size": 5,
    "page_index": 1,
    "ann_type": "A",
    "stock_list": "300750",  # 宁德时代
    "begin_time": "2024-06-01",
    "end_time": "2024-06-30",
}
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://data.eastmoney.com/",
}
resp = requests.get(url, params=params, headers=headers, timeout=30)
data = resp.json()
print(f"Status: {resp.status_code}")
print(f"Keys: {data.keys() if isinstance(data, dict) else 'list'}")
if isinstance(data, dict) and 'data' in data:
    inner = data['data']
    if isinstance(inner, dict):
        print(f"Total pages: {inner.get('total_page')}")
        print(f"Total hits: {inner.get('total_hits')}")
        items = inner.get('list', [])
        for item in items[:3]:
            print(f"  - [{item.get('notice_date')}] {item.get('title','')[:60]}")

## 探索小结

填写以下内容：
- Tushare 连通性: ✅ / ❌
- disclosure 接口可用: ✅ / ❌  
- 试点行业股票数: ___ 只
- 东方财富公告接口可用: ✅ / ❌
- 有什么意外发现：___